# Anthropic Economic Index — Análise do Brasil

Exploração do dataset `Anthropic/EconomicIndex` com foco no uso de IA por ocupação, setor econômico e padrões de automação vs. augmentação no Brasil, comparado a países de perfil similar.

**Dataset:** [Anthropic/EconomicIndex](https://huggingface.co/datasets/Anthropic/EconomicIndex)  
**Release usada:** `release_2025_09_15` (primeira com dados geográficos por país)  
**Data:** Abril 2026

---
### Schema principal — `aei_enriched_claude_ai_*.csv`
| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `geo_id` | str | ISO-3 do país (ex: `BRA`, `USA`) ou código de estado americano |
| `geography` | str | `country`, `state_us` ou `global` |
| `facet` | str | Dimensão de análise: `country`, `onet_task`, `collaboration`, `request`, etc. |
| `variable` | str | Métrica: `usage_pct`, `usage_per_capita_index`, `onet_task_pct`, `automation_pct`, `augmentation_pct`, … |
| `cluster_name` | str | Entidade dentro do facet (ex: nome da tarefa O\*NET, padrão de colaboração) |
| `value` | float | Valor numérico da métrica |

## 1. Setup e Imports

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Países comparáveis ao Brasil (ISO-3)
BRAZIL = 'BRA'
COMPARABLES = ['BRA', 'ARG', 'MEX', 'IND', 'ZAF', 'COL', 'CHL']
LABELS = {'BRA': 'Brasil', 'ARG': 'Argentina', 'MEX': 'México',
          'IND': 'Índia', 'ZAF': 'África do Sul', 'COL': 'Colômbia', 'CHL': 'Chile'}

print('Setup concluído.')

Setup concluído.


## 2. Download do Dataset via Hugging Face

In [6]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID   = 'Anthropic/EconomicIndex'
REPO_TYPE = 'dataset'
RELEASE   = 'release_2025_09_15'

# Lista todos os arquivos da release escolhida
all_files = list(list_repo_files(REPO_ID, repo_type=REPO_TYPE))
release_files = [f for f in all_files if f.startswith(RELEASE)]

print(f'Arquivos em {RELEASE}:')
for f in sorted(release_files):
    print(' ', f)

Arquivos em release_2025_09_15:
  release_2025_09_15/README.md
  release_2025_09_15/code/aei_analysis_functions_1p_api.py
  release_2025_09_15/code/aei_analysis_functions_claude_ai.py
  release_2025_09_15/code/aei_report_v3_analysis_1p_api.ipynb
  release_2025_09_15/code/aei_report_v3_analysis_claude_ai.ipynb
  release_2025_09_15/code/aei_report_v3_change_over_time_claude_ai.py
  release_2025_09_15/code/aei_report_v3_preprocessing_claude_ai.ipynb
  release_2025_09_15/code/preprocess_gdp.py
  release_2025_09_15/code/preprocess_iso_codes.py
  release_2025_09_15/code/preprocess_onet.py
  release_2025_09_15/code/preprocess_population.py
  release_2025_09_15/data/input/BTOS_National.xlsx
  release_2025_09_15/data/input/Population by single age _20250903072924.csv
  release_2025_09_15/data/input/automation_vs_augmentation_v1.csv
  release_2025_09_15/data/input/automation_vs_augmentation_v2.csv
  release_2025_09_15/data/input/bea_us_state_gdp_2024.csv
  release_2025_09_15/data/input/census_st

In [7]:
def download(fname):
    return hf_hub_download(
        repo_id=REPO_ID, filename=fname,
        repo_type=REPO_TYPE, local_dir=DATA_DIR
    )

# Arquivo principal de uso enriquecido (claude.ai)
enriched_file = next(
    f for f in release_files
    if 'enriched_claude_ai' in f and f.endswith('.csv')
)
path_aei = download(enriched_file)
print('AEI enriquecido:', path_aei)

# Dados auxiliares de input (população, GDP, ISO codes)
input_files = [f for f in release_files if '/input/' in f and f.endswith('.csv')]
paths_input = {Path(f).stem: download(f) for f in input_files}
print('Auxiliares:', list(paths_input.keys()))

release_2025_09_15/data/output/aei_enric(…):   0%|          | 0.00/26.8M [00:00<?, ?B/s]

AEI enriquecido: ..\data\release_2025_09_15\data\output\aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv


release_2025_09_15/data/input/Population(…):   0%|          | 0.00/2.18k [00:00<?, ?B/s]

release_2025_09_15/data/input/automation(…):   0%|          | 0.00/197 [00:00<?, ?B/s]

release_2025_09_15/data/input/automation(…):   0%|          | 0.00/198 [00:00<?, ?B/s]

...

release_2025_09_15/data/input/soc_struct(…):   0%|          | 0.00/77.2k [00:00<?, ?B/s]

release_2025_09_15/data/input/task_pct_v(…):   0%|          | 0.00/461k [00:00<?, ?B/s]

release_2025_09_15/data/input/task_pct_v(…):   0%|          | 0.00/435k [00:00<?, ?B/s]

release_2025_09_15/data/input/working_ag(…):   0%|          | 0.00/23.0k [00:00<?, ?B/s]

Auxiliares: ['Population by single age _20250903072924', 'automation_vs_augmentation_v1', 'automation_vs_augmentation_v2', 'bea_us_state_gdp_2024', 'sc-est2024-agesex-civ', 'soc_structure_raw', 'task_pct_v1', 'task_pct_v2', 'working_age_pop_2024_country_raw']


## 3. Exploração Inicial

In [ ]:
df = pd.read_csv(path_aei)

print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Valores únicos das dimensões principais
for col in ['geography', 'facet', 'variable']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().to_string())

In [ ]:
# Cobertura geográfica: quantos países, estados
countries = df[df['geography'] == 'country']['geo_id'].unique()
print(f'Países no dataset: {len(countries)}')
print(f'Brasil presente: {BRAZIL in countries}')

# Período de coleta
print(f"\nPeríodo: {df['date_start'].min()} → {df['date_end'].max()}")

In [ ]:
# Dados de população e GDP para contextualizar
pop_key  = next((k for k in paths_input if 'pop' in k and 'country' in k), None)
gdp_key  = next((k for k in paths_input if 'gdp' in k and 'country' in k), None)
iso_key  = next((k for k in paths_input if 'iso' in k), None)

df_pop = pd.read_csv(paths_input[pop_key])  if pop_key else None
df_gdp = pd.read_csv(paths_input[gdp_key])  if gdp_key else None
df_iso = pd.read_csv(paths_input[iso_key])  if iso_key else None

print('Pop:', df_pop.shape if df_pop is not None else 'não encontrado')
print('GDP:', df_gdp.shape if df_gdp is not None else 'não encontrado')
print('ISO:', df_iso.shape if df_iso is not None else 'não encontrado')

## 4. Brasil vs. Países Comparáveis

### 4.1 Uso per capita indexado

In [ ]:
# usage_per_capita_index: razão entre o uso per capita do país e a média global
# Valores > 1 → uso acima da média; < 1 → abaixo
df_usage = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable']  == 'usage_per_capita_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .sort_values('value', ascending=False)
)

df_usage[['country', 'geo_id', 'value']].rename(columns={'value': 'usage_per_capita_index'})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2196F3' if g == BRAZIL else '#90CAF9' for g in df_usage['geo_id']]
bars = ax.barh(df_usage['country'], df_usage['value'], color=colors)
ax.axvline(1, color='gray', linestyle='--', linewidth=1, label='Média global')
ax.set_xlabel('Índice de uso per capita (média global = 1)')
ax.set_title('Adoção de IA — Uso per capita indexado')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'usage_per_capita_index.png', dpi=150)
plt.show()

### 4.2 Automação vs. Augmentação

In [ ]:
# automation_pct  → tarefas dirigidas pelo modelo sem supervisão humana contínua
# augmentation_pct → tarefas em que o humano mantém controle e validação
df_auto = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(['automation_pct', 'augmentation_pct'])) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(
        country=lambda x: x['geo_id'].map(LABELS),
        variable=lambda x: x['variable'].str.replace('_pct', '')
    )
    .pivot_table(index=['country', 'geo_id'], columns='variable', values='value')
    .reset_index()
    .sort_values('automation', ascending=False)
)

df_auto

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df_auto))
w = 0.35
ax.bar([i - w/2 for i in x], df_auto['automation'],  width=w, label='Automação',   color='#EF5350')
ax.bar([i + w/2 for i in x], df_auto['augmentation'], width=w, label='Augmentação', color='#42A5F5')
ax.set_xticks(list(x))
ax.set_xticklabels(df_auto['country'], rotation=20, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Automação vs. Augmentação por País')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'automation_vs_augmentation.png', dpi=150)
plt.show()

### 4.3 Top tarefas O\*NET no Brasil

In [ ]:
# onet_task_pct: % do uso do país concentrado em cada tarefa O*NET
df_tasks_br = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'onet_task') &
        (df['variable'] == 'onet_task_pct')
    ]
    .nlargest(15, 'value')
    .sort_values('value')
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_tasks_br['cluster_name'], df_tasks_br['value'], color='#1565C0')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Top 15 Tarefas O*NET — Brasil', fontsize=13)
ax.set_xlabel('% do uso total no país')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'brazil_top_onet_tasks.png', dpi=150)
plt.show()

### 4.4 Especialização do Brasil vs. comparáveis (índice de tarefa)

In [ ]:
# onet_task_pct_index: quanto o país é especializado numa tarefa vs. a média global
# Índice > 1 → sobrerrepresentado; < 1 → subrrepresentado
df_idx = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'onet_task') &
        (df['variable']  == 'onet_task_pct_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
)

# Top 10 tarefas mais diferenciadas no Brasil
top_tasks = (
    df_idx[df_idx['geo_id'] == BRAZIL]
    .nlargest(10, 'value')['cluster_name']
    .tolist()
)

df_heat = (
    df_idx[df_idx['cluster_name'].isin(top_tasks)]
    .pivot_table(index='cluster_name', columns='country', values='value')
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    df_heat, annot=True, fmt='.2f', cmap='RdYlGn',
    center=1, linewidths=0.4, ax=ax
)
ax.set_title('Índice de especialização por tarefa O*NET\n(1 = média global)', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_task_specialization.png', dpi=150)
plt.show()

## 5. Visualizações Adicionais

### 5.1 Padrões de colaboração no Brasil (Plotly)

In [ ]:
# collaboration_pct_index: especialização relativa por padrão de interação
# Padrões: directive, feedback_loop, validation, task_iteration, learning
df_collab = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'collaboration') &
        (df['variable'] == 'collaboration_pct_index')
    ]
    .sort_values('value', ascending=False)
)

fig = px.bar(
    df_collab,
    x='cluster_name', y='value',
    title='Brasil — Especialização por padrão de colaboração<br><sup>Índice relativo à média global (1 = média)</sup>',
    labels={'cluster_name': 'Padrão', 'value': 'Índice'},
    color='value',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1
)
fig.add_hline(y=1, line_dash='dash', line_color='gray')
fig.update_layout(coloraxis_showscale=False, xaxis_title='')
fig.write_html(OUTPUT_DIR / 'brazil_collaboration_index.html')
fig.show()

### 5.2 Scatter: uso per capita vs. automação (países comparáveis)

In [ ]:
# Combina usage_per_capita_index + automation_pct para os países comparáveis
df_scatter = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(['usage_per_capita_index', 'automation_pct'])) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .pivot_table(index='geo_id', columns='variable', values='value')
    .reset_index()
    .assign(country=lambda x: x['geo_id'].map(LABELS))
)

# Enriquecer com GDP per capita se disponível
if df_gdp is not None and df_pop is not None:
    df_econ = df_gdp.merge(df_pop, on='iso_alpha_3')
    df_econ['gdp_per_capita'] = df_econ['gdp'] * 1e9 / df_econ['working_age_pop']
    df_scatter = df_scatter.merge(
        df_econ[['iso_alpha_3', 'gdp_per_capita']],
        left_on='geo_id', right_on='iso_alpha_3', how='left'
    )
    size_col = 'gdp_per_capita'
else:
    df_scatter['gdp_per_capita'] = 1
    size_col = 'gdp_per_capita'

fig = px.scatter(
    df_scatter,
    x='usage_per_capita_index', y='automation_pct',
    text='country', size=size_col,
    title='Adoção vs. Automação — Países comparáveis<br><sup>Tamanho = PIB per capita</sup>',
    labels={
        'usage_per_capita_index': 'Índice de uso per capita',
        'automation_pct': 'Taxa de automação'
    },
    color='country',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='top center')
fig.add_vline(x=1, line_dash='dot', line_color='lightgray')
fig.update_layout(showlegend=False, yaxis_tickformat='.0%')
fig.write_html(OUTPUT_DIR / 'scatter_adoption_vs_automation.html')
fig.show()

### 5.3 Treemap interativo — Top tarefas O\*NET no Brasil

In [ ]:
df_tree = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'onet_task') &
        (df['variable'] == 'onet_task_pct')
    ]
    .nlargest(30, 'value')
    # cluster_name pode ser 'grupo::tarefa'; extraímos o grupo
    .assign(
        group=lambda x: x['cluster_name'].str.split('::').str[0],
        task=lambda x:  x['cluster_name'].str.split('::').str[-1]
    )
)

fig = px.treemap(
    df_tree,
    path=['group', 'task'],
    values='value',
    title='Brasil — Top 30 Tarefas O*NET por volume de uso',
    color='value',
    color_continuous_scale='Blues',
    hover_data={'value': ':.2%'}
)
fig.update_traces(textinfo='label+percent entry')
fig.write_html(OUTPUT_DIR / 'brazil_treemap_onet_tasks.html')
fig.show()

## 6. Exportar tabela-resumo

In [ ]:
# Tabela consolidada: métricas-chave por país comparável
metrics = ['usage_per_capita_index', 'usage_pct', 'automation_pct', 'augmentation_pct']

df_summary = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(metrics)) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .pivot_table(index='geo_id', columns='variable', values='value')
    .reset_index()
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .set_index('country')
    .drop(columns='geo_id')
    .sort_values('usage_per_capita_index', ascending=False)
)

df_summary.to_csv(OUTPUT_DIR / 'summary_comparables.csv')
df_summary.style.format({
    'usage_per_capita_index': '{:.2f}',
    'usage_pct':              '{:.2%}',
    'automation_pct':         '{:.2%}',
    'augmentation_pct':       '{:.2%}',
}).background_gradient(cmap='Blues', subset=['usage_per_capita_index'])

## 7. Próximos Passos

- [ ] Comparar evoluções temporais usando `release_2026_01_15` e `release_2026_03_24`
- [ ] Cruzar tarefas O\*NET com dados IBGE/PNAD para estimar exposição por ocupação no Brasil
- [ ] Analisar padrões de uso da API 1P (`aei_raw_1p_api_*.csv`) para contexto empresarial
- [ ] Explorar curvas de aprendizado presentes na release `2026-03-24`